In [1]:
import pandas as pd
import seaborn as sns
from imblearn.over_sampling import SMOTE
from collections import Counter
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report, recall_score, confusion_matrix, average_precision_score, precision_recall_curve, roc_auc_score, cohen_kappa_score

In [2]:
df = pd.read_csv(r"C:\Users\dbastola2022\OneDrive - Florida Atlantic University\Academics\Research\Malnutrition\MICS\malnutrition\Dataset\ch.csv") #Local
df.head(2)

,child_age,child_weight,diarrhoea_last_2_weeks,fever_last_2_weeks,area,child_sex,mother_education,health_insurance,wealth_index,malnurished,province_1.0,province_2.0,province_3.0,province_4.0,province_5.0,province_6.0,province_7.0
0,1,-1.085628,0,0,0,1,5,0,1,1,1,0,0,0,0,0,0
1,3,0.420314,0,1,0,1,5,0,1,1,1,0,0,0,0,0,0


### Train-test split

In [3]:
X = df.drop(columns=['malnurished'])
y = df['malnurished']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, stratify = y, random_state=42)

In [4]:
sm = SMOTE(random_state=42)
X_train_sm, y_train_sm = sm.fit_resample(X_train, y_train)
print('Before SMOTE: ', Counter(y_train))
print('After SMOTE: ', Counter(y_train_sm))

Before SMOTE:  Counter({0: 2828, 1: 2316})
After SMOTE:  Counter({1: 2828, 0: 2828})


# Support Vector Machine

### Simple model

In [5]:
svm = SVC(probability=True, random_state=42)
svm.fit(X_train_sm, y_train_sm)

SVC(probability=True, random_state=42)

In [6]:
y_pred = svm.predict(X_test)
print(classification_report(y_test, y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.789     0.741     0.764       707
           1      0.706     0.758     0.731       579

    accuracy                          0.749      1286
   macro avg      0.747     0.750     0.748      1286
weighted avg      0.752     0.749     0.749      1286



In [7]:
y_probas = svm.predict_proba(X_test)[:, 1]
print(f'Average Precision: {average_precision_score(y_test, y_probas)}')

Average Precision: 0.7996660709775093


## Hyperparameter Tuning

In [8]:
params = {
    'C': [0.01, 0.1, 1, 2],
    'kernel': ['linear', 'rbf', 'poly'],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1, 1],
}

grid_search = GridSearchCV(
    estimator=SVC(probability=True, random_state=42),
    param_grid=params,
    scoring='recall',
    cv=10,
    verbose=1,
    n_jobs=-1
)

grid_search.fit(X_train_sm, y_train_sm)

Fitting 10 folds for each of 72 candidates, totalling 720 fits


KeyboardInterrupt: 

In [ ]:
print("Best Parameters:", grid_search.best_params_)
print(f"Best cross-validation recall score: {grid_search.best_score_:.3f}")

Best Parameters: {'C': 0.01, 'gamma': 'scale', 'kernel': 'linear'}
Best cross-validation recall score: 0.744


In [ ]:
svm_tune = grid_search.best_estimator_
y_pred_tune = svm_tune.predict(X_test)
print("Classification Report:\n", classification_report(y_test, y_pred_tune, digits=3))

Classification Report:
               precision    recall  f1-score   support

           0      0.778     0.710     0.743       707
           1      0.680     0.753     0.715       579

    accuracy                          0.729      1286
   macro avg      0.729     0.732     0.729      1286
weighted avg      0.734     0.729     0.730      1286



### Average precision

In [ ]:
y_probas_tune = svm_tune.predict_proba(X_test)[:, 1]
print(f'Average Precision: {average_precision_score(y_test, y_probas_tune)}')

AttributeError: predict_proba is not available when probability=False

In [ ]:
display = PrecisionRecallDisplay.from_estimator(
    svm_tune,
    X_test,
    y_test, 
    name="SVM",
    plot_chance_level = True,
)
display.ax_.set_title('2-class Precision Recall Curve')
plt.show()

NameError: name 'PrecisionRecallDisplay' is not defined

### Recall score on train set

In [ ]:
# Recall on base model
y_train_pred = svm.predict(X_train_sm)
train_recall = recall_score(y_train_sm, y_train_pred)
print(f"Recall on Training set (Base Model): {train_recall:.3f}")

# Recall on tune tune model
y_train_pred = svm_tune.predict(X_train_sm)
train_recall = recall_score(y_train_sm, y_train_pred)
print(f"Recall on Training set (Tune Model): {train_recall:.3f}")

Recall on Training set (Base Model): 0.743
Recall on Training set (Tune Model): 0.744


### AUC score

In [ ]:
auc = roc_auc_score(y_test, y_probas)
print(f"AUC: {auc:.3f}")

auc_tune = roc_auc_score(y_test, y_probas_tune)
print(f"AUC: {auc_tune:.3f}")

AUC: 0.805


NameError: name 'y_probas_tune' is not defined

### Cohen's kappa score

In [ ]:
kappa = cohen_kappa_score(y_test, y_pred)
print(f"Cohen's Kappa: {kappa:.3f}")

kappa_tune = cohen_kappa_score(y_test, y_pred_tune)
print(f"Cohen's Kappa: {kappa_tune:.3f}")

### Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred_tune)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Nurished', 'Malnurished'], yticklabels=['Nurished', 'Malnurished'])
plt.title('Confusion Matrix')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

### Feature importance